
# End-to-End Batch Pipeline

## Business Scenario
Management requires daily regional revenue, SLA breach %, and usage summary for operational dashboards.

## Problem Statement
Design full batch pipeline:

READ → TRANSFORM → AGGREGATE → WRITE

## Business Requirements
- Compute total revenue per region
- Compute SLA breach percentage
- Aggregate by service_type and plan_type
- Persist results in Parquet format


In [4]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, avg, count

spark = SparkSession.builder.appName("e2e_pipeline_demo").getOrCreate()
df = spark.read.csv("telecom_cdr_production.csv", header=True, inferSchema=True)

aggregated_df = df.groupBy("region")     .agg(
        sum("revenue").alias("total_revenue"),
        avg("sla_breach_flag").alias("sla_breach_ratio"),
        count("*").alias("total_events")
    )

aggregated_df.show(5)

#aggregated_df.write.mode("overwrite").parquet("regional_kpi_output")


+------+-----------------+-------------------+------------+
|region|    total_revenue|   sla_breach_ratio|total_events|
+------+-----------------+-------------------+------------+
| South|986374.5199998763| 0.4436312388395538|      124883|
|  East|993644.7699998801|0.44443025344242665|      125275|
|  West|983410.4499998775|0.44413983278423075|      124749|
| North|991092.0299998743|0.44544458922561614|      125093|
+------+-----------------+-------------------+------------+



In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import desc

spark = SparkSession.builder.appName("e2e_pipeline_demo").getOrCreate()
df = spark.read.csv("telecom_cdr_production.csv", header=True, inferSchema=True)

# Top customers
top_customers = df.groupBy("customer_id").sum("revenue").orderBy(desc("sum(revenue)")).limit(10)

top_customers.show(5)

# Avg latency by service
avg_latency = df.groupBy("service_type").avg("sla_latency_ms")

avg_latency.show(5)

# Revenue by roaming
revenue_roaming = df.groupBy("roaming_type").sum("revenue")

revenue_roaming.show(5)

+-----------+------------------+
|customer_id|      sum(revenue)|
+-----------+------------------+
|      33699|385.78999999999996|
|      34425|341.46000000000004|
|      23101|            328.35|
|      49180|324.04999999999995|
|      17125|            317.95|
+-----------+------------------+
only showing top 5 rows

+------------+-------------------+
|service_type|avg(sla_latency_ms)|
+------------+-------------------+
|         SMS| 274.83095271030174|
|        DATA|  275.0487823910395|
|       VOICE| 275.24690360796984|
+------------+-------------------+

+-------------+-----------------+
| roaming_type|     sum(revenue)|
+-------------+-----------------+
|INTERNATIONAL|2371262.200000394|
|         HOME|792519.3199998505|
|     NATIONAL| 790740.249999851|
+-------------+-----------------+

